In [ ]:
import pathlib
import platform
import psutil
import torch
import torchvision
from PIL import Image

print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"Platform:     {platform.platform()}")

device = "mps"

In [ ]:
COCO4000_ROOT = pathlib.Path("../datasets/coco4000")
IMAGES_DIR = COCO4000_ROOT / "images" / "train2017"
LABELS_DIR = COCO4000_ROOT / "labels" / "train2017"

transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((640, 640)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)


class COCOClassificationDataset(torch.utils.data.Dataset):
    """Reads COCO4000 images and extracts the first class_id from each YOLO label file."""

    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = pathlib.Path(images_dir)
        self.labels_dir = pathlib.Path(labels_dir)
        self.transform = transform
        self.samples = sorted(self.images_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label_path = self.labels_dir / (img_path.stem + ".txt")
        label = 0
        if label_path.exists():
            text = label_path.read_text().strip()
            if text:
                label = int(text.split()[0]) + 1

        return image, label


dataset = COCOClassificationDataset(IMAGES_DIR, LABELS_DIR, transform=transform)
train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=64, shuffle=True, num_workers=0, pin_memory=False
)

In [ ]:
NUM_CLASSES = 80 + 1  # extra class for no objects

model = torchvision.models.mobilenet_v3_small(weights=None)

# freeze first part of model
for i, block in enumerate(model.features):
    if i < 12:
        for param in block.parameters():
            param.requires_grad = False

# replace classifier head for COCO classes
in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.01,
    momentum=0.9,
)

In [ ]:
def get_comp_state():
    mps_alloc = torch.mps.current_allocated_memory() / 1e9
    mps_driver = torch.mps.driver_allocated_memory() / 1e9
    ram = psutil.virtual_memory()
    ram_used = (ram.total - ram.available) / 1e9
    ram_total = ram.total / 1e9
    swap = psutil.swap_memory()
    swap_used = swap.used / 1e9
    return f"MPS alloc= {mps_alloc:.2f} GB | MPS driver= {mps_driver:.2f} GB | RAM= {ram_used:.1f}/{ram_total:.1f} GB ({ram.percent:.1f}%) | swap= {swap_used:.2f} GB"


EPOCHS = 3

print(f"BEFORE TRAINING: {get_comp_state()}\n")

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1} START: {get_comp_state()}")
    model.train()

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        if (batch_idx + 1) % 5 == 0:
            print(
                f"   Epoch {epoch + 1} Batch {batch_idx + 1}: {get_comp_state()}",
                flush=True,
            )

    # Flush the MPS command buffer so reported memory numbers are accurate
    torch.mps.synchronize()

    print(f"Epoch {epoch + 1} END:   {get_comp_state()}")

print(f"\nAFTER TRAINING:  {get_comp_state()}")